In [ ]:
%cd ..

In [1]:
# !pip install trino --proxy 192.168.5.8:3128
# !pip install jaydebeapi JPype1 --proxy 192.168.5.8:3128
# !pip install psycopg2-binary pymysql --proxy 192.168.5.8:3128

In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
import sys
import os

sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
from common.config import *
from common.http_util import  *
from common.crawler_util import  *
from common.ambari_util import *
import requests
import time
import json
from datetime import datetime, timedelta

In [ ]:
# ================= CONFIG =================
FRESHDESK_DOMAIN = "https://vcs-care.freshdesk.com"
SEARCH_URL = f"{FRESHDESK_DOMAIN}/api/v2/search/tickets"

FRESHDESK_API_KEY=os.getenv("FRESHDESK_API_KEY")

if not FRESHDESK_API_KEY:
    raise Exception("Not found FRESHDESK_API_KEY")

HEADERS = {
    "Authorization": build_basic_auth_header(FRESHDESK_API_KEY, "X"),
}

MAX_PAGE = 10
PAGE_SIZE = 30
MAX_RETRY = 5
RETRY_SLEEP = 10
NORMAL_SLEEP = 1
# =========================================

In [ ]:
import trino
from trino.auth import BasicAuthentication
import jaydebeapi

TRINO_URL = os.getenv("TRINO_URL")
TRINO_USER = os.getenv("TRINO_USER")
TRINO_PASSWORD = os.getenv("TRINO_PASSWORD")

def get_existing_ticket_ids():
    conn = trino.dbapi.connect(
        host="10.254.130.48",
        port=8090,
        http_scheme="https",
        user="crawler_service",
        auth=BasicAuthentication("user", "password"),
        catalog="hive",
        schema="freshdesk"
    )

    cur = conn.cursor()
    cur.execute("""
       select id, status from hive.cx_cso_raw.fact_cso_tickets
    """)

    # fetchall trả list tuple
    return {str(row[0]) for row in cur.fetchall()}

def get_existing_ticket_ids2():
    TRINO_JDBC_JAR = r"C:\Users\namtv40\Libs\trino-jdbc-479.jar"

    jdbc_url = TRINO_URL

    props = {
        "user": TRINO_USER,
        "password": TRINO_PASSWORD,
        "SSL": "false"
    }

    conn = jaydebeapi.connect(
        "io.trino.jdbc.TrinoDriver",
        jdbc_url,
        props,
        TRINO_JDBC_JAR
    )

    cursor = conn.cursor()
    cursor.execute("""
       select id, status from hive.cx_cso_raw.fact_cso_tickets
    """)

    ticket_ids = {str(row[0]) for row in cursor.fetchall()}

    cursor.close()
    conn.close()
    return ticket_ids
    

In [ ]:
def fetch_resource_name_freshdesk(records):
    resource_name = "fact_cso_tickets"
    print("Start crawl : ", resource_name)
    start_time= time.time()

    HDFS_BASE = "s3a://vcs-raw/cx-cso-raw"
    STATE_PATH = "s3a://vcs-raw/cx-cso-raw/crawler/state"
    BASE_URL = "https://vcs-care.freshdesk.com/api/v2"
    RESOURCE_URL = "search/tickets"
    API_KEY_PATH = "s3a://vcs-raw/cx-cso-raw/crawler/envs/api_key.txt"
    API_COOKIE_PATH = "/user/secure/freshdesk/api_cookie_key.txt"
    QUERY_PARAMS = {
      "page": 1,
      "include": "company",
    }
    ENABLE_STATE = False
    hive_db = "fact_cso_tickets"
    crawl_mode = "large_and_modified_and_new"
    schema_local_path = "./resources/parquet_schema/fact_cso_tickets.json"

    print(resource_name)
    start_time= time.time()
    # =========================
    # MAIN
    # =========================

    if ENABLE_STATE:
        last_state = read_last_state(resource_name)
        print("Last state =", last_state)
    else:
        last_state = None

    if not records:
        print("No new data")
        out_of_data = True
        return True

    # =========================
    # Pandas → Parquet
    # =========================

    now = datetime.now()
    partition_path = "{}/{}".format(
        HDFS_BASE,
        resource_name
    )

    filename = "data_{}_{}{:02d}{:02d}_{}{:02d}{:02d}.parquet".format(
        resource_name,
        now.year,
        now.month,
        now.day,
        now.hour,
        now.minute,
        now.second
    )
    local_parquet = "./tmp/data/cx_cso_raw/{}/{}".format(resource_name,filename)
    os.makedirs(os.path.dirname(local_parquet), exist_ok=True)

    # df.to_parquet(local_parquet,engine="pyarrow", compression="snappy", index=False)
    records = convert_json_add_ts_columns(records)

    schema_tm_path = "./resources/parquet_schema/cx_cso_raw/{}.json".format(resource_name)
    schema = None
    if schema_local_path:
        schema = load_pyarrow_schema_from_json(schema_local_path)

    if os.path.exists(schema_tm_path):
        schema = load_pyarrow_schema_from_json(schema_tm_path)

    if not schema:
        schema = infer_schema_from_json(records)
        schema_json = save_pyarrow_type_to_json(schema)
        write_file_json(schema_tm_path, schema_json)
        
    records = convert_json_list_by_arrow_schema(records, schema)

    df = pd.DataFrame(records)
    data_table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)

    pq.write_table(data_table, local_parquet, compression="snappy")

    upload_hdfs_https(
        "{}".format(partition_path),
        local_parquet
    )

    print("Uploaded parquet to", partition_path)

    # =========================
    # Generate SQL (TEXT ONLY)
    # =========================
    sql = gen_spark_create_table(schema=schema,db=hive_db, table=resource_name, location="{}/{}".format(HDFS_BASE, resource_name))

    # filename = "create_table_{}_{}{:02d}{:02d}.sql".format(
    #     resource_name,
    #     now.hour,
    #     now.minute,
    #     now.second
    # )

    filename = "create_table_{}.sql".format(
        resource_name
    )

    local_sql = "./tmp/data/cx_cso_raw/{}/{}".format(resource_name, filename)
    os.makedirs(os.path.dirname(local_sql), exist_ok=True)

    with open(local_sql, "w") as f:
        f.write(sql)

    # upload_hdfs_https(
    #     "{}/{}".format(HDFS_BASE, resource_name),
    #     local_sql
    # )

    print("Uploaded SQL definition")

    # =========================
    # Save new state
    # =========================
    if ENABLE_STATE and "updated_at" in df.columns:
        max_ts = get_max_updated_at_str(df)
        max_ts = subtract_minutes(max_ts, 900)
        write_last_state(max_ts, resource_name)

    elapsed = time.time() - start_time
    print("Loop {} took {:.3f}s".format(resource_name, elapsed))
    if len(records) < 100:
        print("No new data")
        out_of_data = True
        return True
    return False

In [6]:
def fetch_tickets_by_date(date_str):
    """
    Crawl tất cả ticket trong 1 ngày (có phân trang)
    """
    all_results = []

    for page in range(1, MAX_PAGE + 1):
        params = {
            "query": f"\"created_at:'{date_str}'\"",
            # "query": f"\"updated_at:'{date_str}'\"",
            "page": page,
            "include": "company",
        }
        print(f"\nCrawling date: {date_str} at round {page}")
        retry = 0
        for attempt in range(MAX_RETRY):
            try:
                print("Request to ", SEARCH_URL , params)
                resp = requests.get(
                        SEARCH_URL,
                        headers=HEADERS,
                        params=params,
                        proxies=PROXIES,
                        timeout=30
                    )
                
                if resp.status_code == 429:
                    wait = RETRY_SLEEP * attempt
                    print(f"[429] Rate limit {date_str} page={page}, retry {attempt}, sleep {wait}s")
                    time.sleep(wait)
                    continue
                
                resp.raise_for_status()
                data = resp.json()
                results = data.get("results", [])

                all_results.extend(results)

                # hết data thì break sớm
                if len(results) < PAGE_SIZE:
                    return all_results
                break  # page OK → sang page tiếp theo
            except requests.exceptions.Timeout:
                print("\n⚠️ Timeout khi gọi {}. Thử lại {}/{}...".format(SEARCH_URL, attempt + 1, MAX_RETRY))
                if attempt < MAX_RETRY - 1:
                    time.sleep(10)
            except requests.exceptions.ProxyError as e:
                print("\n❌ ProxyError khi gọi {}: {}".format(SEARCH_URL, e))
                if attempt < MAX_RETRY - 1:
                    time.sleep(10)
                else:
                    print(f"[ERROR] {date_str} page={page} - {resp.status_code} - {resp.text}")
                    return all_results
            except requests.exceptions.RequestException as e:
                print("\n❌ Lỗi request khi gọi {}: {}".format(SEARCH_URL, e))
                if attempt < MAX_RETRY - 1:
                    time.sleep(10)
                else:
                    print(f"[ERROR] {date_str} page={page} - {resp.status_code} - {resp.text}")
                    return all_results
        time.sleep(NORMAL_SLEEP)

    return all_results


def crawl_until_today(start_date: str):
    """
    Crawl từ start_date đến ngày hiện tại
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    today = datetime.now().date()

    all_tickets = []

    current = start
    while current.date() <= today:
        date_str = current.strftime("%Y-%m-%d")
        print(f"\nCrawling date: {date_str}")

        tickets = fetch_tickets_by_date(date_str)
        print(f" -> Found {len(tickets)} tickets")

        all_tickets.extend(tickets)
        current += timedelta(days=1)

    return all_tickets


def fetch_ticket_detail(ticket_id, retry=5):
    url = f"{FRESHDESK_DOMAIN}/api/v2/tickets/{ticket_id}"
    params = {
        "include": "company",
    }
    for i in range(retry):
        try:
            r = requests.get(
                url,
                headers=HEADERS,
                params=params,
                proxies=PROXIES,
                timeout=30
            )
            if r.status_code == 200:
                return r.json()

            if r.status_code == 429:
                time.sleep(5 * (i + 1))
                continue
            else:
                print(f"[ERROR] ticket_id={ticket_id} {r.status_code}")
        except :
            continue
        return None
    print(f"[ERROR] ticket_id={ticket_id}")
    return None

def fetch_missing_ticket_ids(start_id, end_id, existed_ids):
    records = []
    for ticket_id in range(start_id, end_id):
        if ticket_id %10 == 0:
            print(f"Processed at {ticket_id}")
        
        ticket_id  = str(ticket_id)
        if ticket_id in existed_ids:
            continue
        row = fetch_ticket_detail(ticket_id, retry=5)
        if not row:
            continue
        records += [row]
        
    return records

In [7]:
start_date = "2025-12-30"
today = datetime.now().date()
start_date = today.strftime("%Y-%m-%d")

# tickets = crawl_until_today(start_date)
# if tickets:
#     print(f"\nTOTAL tickets: {len(tickets)} at {start_date} to now")

#     with open(f"./tmp/fact_cso_tickets/freshdesk_tickets_until_today_{start_date}.json", "w", encoding="utf-8") as f:
#         json.dump(tickets, f, ensure_ascii=False, indent=2)
        
#     fetch_resource_name_freshdesk(tickets)

In [8]:
existed_ids = get_existing_ticket_ids2()

In [9]:
len(existed_ids)

58127

In [10]:
max([int(id) for id in existed_ids])

58560

In [11]:
#58338
filename = "./tmp/fact_cso_tickets-last-id.json"
if os.path.exists(filename):
    start_id = read_file_json(filename)
    start_id = int(start_id)
else:
    start_id = max([int(id) for id in existed_ids]) - 600
end_id = max([int(id) for id in existed_ids])
print("start_id=",start_id)
print("end_id", end_id)
time.sleep(2)
delta_size = 100
while start_id < end_id:
    current_start_id = start_id
    current_end_id = start_id + delta_size
    if current_end_id > end_id:
        current_end_id = end_id
    records = fetch_missing_ticket_ids(current_start_id,current_end_id, existed_ids)
    if not records:
        print("Not found records ", start_date, current_start_id)
        start_id = start_id + delta_size
        time.sleep(2)
        continue
    with open(f"./tmp/fact_cso_tickets/freshdesk_tickets_missing_{start_date}_{current_start_id}.json", "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print("records=", len(records))
    fetch_resource_name_freshdesk(records)
    start_id = start_id + delta_size
    time.sleep(2)

write_file_json(filename, str(end_id))

start_id= 58548
end_id 58560
Processed at 58550
records= 2
Start crawl :  fact_cso_tickets
fact_cso_tickets
Add Upload  s3a://vcs-raw/cx-cso-raw/fact_cso_tickets ./tmp/data/fact_cso_tickets/data_fact_cso_tickets_20260105_131415.parquet
Request to  https://datalake.viettelcyber.com/gateway/ui/ambari/api/v1/views/FILES/versions/1.0.0/instances/FILES/resources/files/upload None
Uploaded parquet to s3a://vcs-raw/cx-cso-raw/fact_cso_tickets
Uploaded SQL definition
Loop fact_cso_tickets took 0.425s
No new data


In [12]:
# for i in range(58000, 59500, 100):
#     print(i)
#     with open(f"./tmp/fact_cso_tickets/freshdesk_tickets_missing_2026-01-05_{i}.json", "r", encoding="utf-8") as f:
#         records = json.load(f)
#         print(len(records))
#     fetch_resource_name_freshdesk(records)
#     time.sleep(2)

In [ ]:
import psycopg2

POSTGRESQL_HOST = os.getenv("POSTGRESQL_HOST")
POSTGRESQL_PORT = os.getenv("POSTGRESQL_PORT")
POSTGRESQL_DATABASE = os.getenv("POSTGRESQL_DATABASE")
POSTGRESQL_USER = os.getenv("POSTGRESQL_USER")
POSTGRESQL_PASSWORD = os.getenv("POSTGRESQL_PASSWORD")

conn = psycopg2.connect(
    host=POSTGRESQL_HOST,
    port=int(POSTGRESQL_PORT),
    database=POSTGRESQL_DATABASE,
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD
)

cur = conn.cursor()

sql = """
select distinct id from cxcso.fact_cso_tickets fct
"""

cur.execute(sql)

rows = cur.fetchall()
pso_id_df_ids = [row[0] for row in rows]
cur.close()
conn.close()

In [14]:
# pso_id_df = pd.read_csv(r"C:\Users\namtv40\fact_cso_tickets_202601051134.csv")
# pso_id_df_ids = pso_id_df["id"].values
# pso_id_df_ids = [int(id) for id in pso_id_df_ids]

In [16]:
len(existed_ids), len(pso_id_df_ids)

(58127, 57988)

In [17]:
records = []
for id in pso_id_df_ids:
    ticket_id = str(id)
    if ticket_id not in existed_ids:
        ticket_id  = str(ticket_id)
        print(ticket_id)
        row = fetch_ticket_detail(ticket_id, retry=5)
        if not row:
            print("not found ", ticket_id)
            continue
        records += [row]
        if len(records) == 100:
            print("Upload partial records")
            print("records=", len(records))
            fetch_resource_name_freshdesk(records)
            records = []
            time.sleep(2)
            
if records:
    print("Upload  final records")
    print("records=", len(records))
    fetch_resource_name_freshdesk(records)
    time.sleep(2)

52949
[ERROR] ticket_id=52949 404
not found  52949


In [18]:
"57582" in existed_ids

True

In [19]:
57582 in pso_id_df_ids

False

In [20]:
57582 in list(pso_id_df_ids)

False

In [21]:
list(pso_id_df_ids)[0]

'1611'